In [26]:
import pandas as pd
from pathlib import Path

tickers_path = Path.cwd().resolve().parent.parent / "Data" / "tickers.csv"

tickers_df = pd.read_csv(tickers_path)
tickers_df

,tickers,Company Name
0,ARG.TO,Amerigo Resources Ltd
1,AYA.TO,Aya Gold & Silver Inc
2,CG.TO,Centerra Gold Inc
3,CGC.V,Canadian Gold Corp
4,CKG.V,Chesapeake Gold Corp
5,DMX.V,District Metals Corp
6,DV.V,Dolly Varden Silver Corp
7,EDG.V,Endurance Gold Corp
8,ELO.TO,Eloro Resources Ltd
9,EQX,Equinox Gold Corp


In [34]:
# necessary imports
import os
import time
import json as _json
from datetime import datetime

from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

BASE_SEARCH_URL = "https://finance.yahoo.com"


# -----------------------------
# Cookies
# -----------------------------
def accept_cookies_if_present(driver, timeout: int = 8):
    """
    Accept Yahoo Finance cookie banner if it appears.
    You may need to tweak the selector or use text like 'Accept all'.
    """
    try:
        wait = WebDriverWait(driver, timeout)
        # Example: generic "Accept all" button on many consent banners
        cookie_button = wait.until(
            EC.element_to_be_clickable((
                By.CSS_SELECTOR,
                "#consent-page > div > div > div > form > div.wizard-body > div.actions.couple > button.btn.secondary.accept-all"
            ))
        )
        cookie_button.click()
        time.sleep(1)
    except TimeoutException:
        # No cookie banner or different layout – just continue
        pass


# -----------------------------
# Driver creation
# -----------------------------
def create_driver(headless: bool = False) -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )


# -----------------------------
# Search company
# -----------------------------
def yahoo_search_company(driver, ticker_company: str):
    """
    1) Open Yahoo Finance homepage
    2) Accept cookies
    3) Type ticker in search bar
    4) Click first result that looks like a quote page (/quote/...)
    """
    driver.get(BASE_SEARCH_URL)
    wait = WebDriverWait(driver, 10)

    accept_cookies_if_present(driver)

    # Search bar
    search_input = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "#ybar-sbq"))
    )
    search_input.clear()
    search_input.send_keys(ticker_company)
    search_input.send_keys(Keys.RETURN)

    # Wait for quote page to load
    wait.until(EC.url_contains("/quote/"))


# -----------------------------
# Collect article URLs (from News tab)
# -----------------------------
def _collect_article_urls(
    driver,
    max_scrolls: int = 25,
    pause: float = 4 # seconds to wait after each scroll
):
    """
    Scroll the news page to load more items and return unique article URLs.
    """
    collected = []
    seen = set()
    last_height = 0

    for i in range(max_scrolls):
        time.sleep(pause)

        # 1) Select ALL news anchors inside the list
        article_links = driver.find_elements(
            By.CSS_SELECTOR,
            "#main-content-wrapper section.mainContent ul li section a[href]"
        )

        # 2) Collect unique URLs
        for link in article_links:
            url = link.get_attribute("href")
            if not url:
                continue
            if url in seen:
                continue
            if "/news/" not in url:
                continue  # skip non-news links
            
            seen.add(url)
            collected.append(url)

        # Debug (optional)
        print(f"Scroll {i}: total unique URLs so far = {len(collected)}")

        # 3) Scroll down to load more
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            # Page didn't grow any further – no more content to load
            break
        last_height = new_height

    return collected


# -----------------------------
# Scrape news URLs for a ticker
# -----------------------------
def scrape_news(driver, ticker: str):
    """
    Assumes you are on the ticker's main quote page.
    Click the 'News' tab, then collect article URLs from that tab.
    """
    wait = WebDriverWait(driver, 10)

    # Click the "News" tab in the quote page navigation.
    # You may need to tweak this selector after inspecting the HTML:
    try:
        news_button = wait.until(
            EC.element_to_be_clickable((
                By.CSS_SELECTOR,
                "#nimbus-app > main > section > section > aside > section > nav > ul > li:nth-child(2) > a"
            ))
        )
        news_button.click()
    except TimeoutException:
        print(f"Could not find News tab for {ticker}")
        return []

    # Now we're on the news tab; collect URLs
    article_urls = _collect_article_urls(driver)
    return article_urls

# -----------------------------------
# Scrape article content
# -----------------------------------
def scrape_article_content(driver, url: str, timeout: int = 8) -> dict:
    """
    Open a Yahoo Finance news article URL and extract:
      - url
      - title
      - published time
      - full text (joined paragraphs)
    If anything fails, fields may be empty strings.
    """
    wait = WebDriverWait(driver, timeout)
    driver.get(url)

    if button := driver.find_elements(By.CSS_SELECTOR, "#main-content-wrapper > div > article > div.body-wrap.yf-tt7to2 > div > div.readmore.active.yf-103i3cu > button"):
        button[0].click()
        time.sleep(1)

    # Title
    try:
        title_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, ".cover-title"))
        )
        title = title_el.text.strip()
    except TimeoutException:
        title = ""

    # Published time (often in <time> element)
    try:
        published = driver.find_element(By.CSS_SELECTOR, ".byline-attr-meta-time").get_attribute("datetime") 
    except Exception:
        published = ""

    # Article text: paragraphs inside article tag
    text = ""
    try:
        paragraphs = driver.find_elements(By.CSS_SELECTOR, ".bodyItems-wrapper .yf-1090901")
        text_parts = [p.text.strip() for p in paragraphs if p.text.strip()]
        text = "\n".join(text_parts)
    except Exception:
        pass

    return {
        "url": url,
        "title": title,
        "published": published,
        "text": text,
    }

# -----------------------------
# Save JSONL
# -----------------------------
def save_jsonl(records, path: str, mode: str = "w"):
    """
    Save one JSON object per line.
    mode="w" overwrites, mode="a" appends.
    """
    with open(path, mode, encoding="utf-8") as f:
        for rec in records:
            f.write(_json.dumps(rec, ensure_ascii=False) + "\n")


# -----------------------------
# Main
# -----------------------------
def main():
    driver = create_driver(headless=False)

    try:
        for ticker in tickers_df["tickers"].tolist()[10:11]:
            print(f"Searching for company: {ticker}")
            yahoo_search_company(driver, ticker)

            urls = scrape_news(driver, ticker)
            print(f"Found {len(urls)} articles for {ticker}")

            records = []
            for url in urls:
                article = scrape_article_content(driver, url)
                article["ticker"] = ticker   
                records.append(article)

            # Save one file per ticker (overwrite if re-run)
            out_path = Path.cwd().resolve().parent.parent / "Data" / "Textual_data" / "Yahoo_articles" 
            out_path = f"{out_path}/yahoo_articles_{ticker}.jsonl"
            save_jsonl(records, out_path, mode="w")
            print(f"Saved {len(records)} articles to {out_path}")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

Searching for company: ESK.V
Scroll 0: total unique URLs so far = 18
Scroll 1: total unique URLs so far = 18
Found 18 articles for ESK.V
Saved 18 articles to /Users/douwepauwels/Downloads/Big Data/Groupwork/Big_data_group_2/Data/Textual_data/Yahoo_articles/yahoo_articles_ESK.V.jsonl


In [10]:
tickers_df["tickers"].tolist()[4:5]

['CKG.V']